# 👥 Atomic Step 3: Speaker Diarization (Pyannote 4.x)

This notebook runs speaker diarization (identifying who spoke when) on the noise-suppressed cleaned audio file using the **Pyannote 4.x community-1** diarization model, and saves the raw segments to JSON.

In [ ]:
# 1. Uninstall incompatible torchao to prevent conflicts
!pip uninstall -y -q torchao

# 2. Install Pyannote 4.x and soundfile
!pip install -q "pyannote.audio>=4.0.1" soundfile --prefer-binary
print("[SUCCESS] Dependencies installed!")

In [ ]:
import torch
from google.colab import userdata

# Retrieve HF Token from Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    print("[SUCCESS] Retrieved Hugging Face access token (HF_TOKEN) from Secrets.")
except Exception:
    hf_token = None
    print("[WARNING] 'HF_TOKEN' not found in Secrets. If Pyannote loading fails, configure it in Colab Secrets.")

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
try:
    drive.mount('/content/drive')
    print("[SUCCESS] Google Drive mounted.")
except Exception:
    print("[INFO] Already mounted or skipped.")

## ⚙️ Parameters

In [ ]:
# @markdown ### 📁 Audio & Output Paths
cleaned_audio_path = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Atomic Notebooks/Cleaned_Audio/MarauliKhurad1_cleaned.wav" # @param {type:"string"}
diarization_output_folder = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Atomic Notebooks/Diarization_Outputs/" # @param {type:"string"}

# @markdown ### 👥 Diarization Clustering Constraints (Optional)
# @markdown Set to 0 to let Pyannote estimate the number of speakers automatically (highly recommended for generic files).
num_speakers = 0 # @param {type:"integer"}
min_speakers = 0 # @param {type:"integer"}
max_speakers = 0 # @param {type:"integer"}

if not os.path.exists(cleaned_audio_path):
    print(f"[ERROR] Cleaned audio not found at: '{cleaned_audio_path}'")
else:
    audio_filename = os.path.basename(cleaned_audio_path)
    audio_name_only = audio_filename.replace("_cleaned.wav", "").replace(".wav", "")
    os.makedirs(diarization_output_folder, exist_ok=True)
    print(f"[SUCCESS] Validated path. Outputs will be saved in: {diarization_output_folder}")

## 👥 Run Diarization

In [ ]:
from pyannote.audio import Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook
import json

if os.path.exists(cleaned_audio_path):
    print("Initializing Pyannote Speaker Diarization Pipeline...")
    try:
        diarization_pipeline = Pipeline.from_pretrained(
            "pyannote/speaker-diarization-community-1",
            token=hf_token
        )
        diarization_pipeline.to(device)
        print("[SUCCESS] Diarization pipeline loaded successfully.")
    except Exception as e:
        print(f"[ERROR] Failed to load Pyannote pipeline. Verify your HF_TOKEN is valid and that you accepted the user conditions on Hugging Face.")
        raise e

    print("\n--- Running Speaker Diarization on Cleaned Audio ---")
    
    # Setup pipeline params
    diarization_params = {}
    if num_speakers > 0:
        diarization_params["num_speakers"] = num_speakers
    else:
        if min_speakers > 0:
            diarization_params["min_speakers"] = min_speakers
        if max_speakers > 0:
            diarization_params["max_speakers"] = max_speakers

    with ProgressHook() as hook:
        diarization_output = diarization_pipeline(cleaned_audio_path, hook=hook, **diarization_params)

    # Extract speaker segments
    raw_speaker_segments = []
    for turn, speaker in diarization_output.speaker_diarization:
        raw_speaker_segments.append({
            "start": turn.start,
            "end": turn.end,
            "speaker": speaker
        })

    print(f"\nDiarization complete. Identified {len(set(s['speaker'] for s in raw_speaker_segments))} unique speaker(s).")
    
    # Save raw segments to JSON file
    raw_json_output_path = os.path.join(diarization_output_folder, f"{audio_name_only}_raw_timeline.json")
    with open(raw_json_output_path, "w", encoding="utf-8") as f:
        json.dump(raw_speaker_segments, f, indent=4, ensure_ascii=False)
        
    print(f"[SUCCESS] Raw speaker timeline exported to: '{raw_json_output_path}'")